# Rock phy py

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import lasio
import rockphypy as rp
from rockphypy import GM, EM
from rockphypy import QI

In [2]:
# parameters
Dqz, Kqz, Gqz = 2.65, 36.6, 45 ## grain density, bulk and shear modulus
Dsh, Ksh, Gsh = 2.7, 21, 7 # shale/clay density, bulk and shear modulus
Dc,Kc, Gc =2.65, 36.6, 45 # cement density, bulk and shear modulus
Db, Kb = 1, 2.2 # brine density, bulk modulus
phi_c=0.4 # critical porosity
sigma=20 # effective pressure
scheme=2
Cn=8.6
vsh=0 # shale volume
# define cement porosity for Vp
phib=0.3
f= 0.5 # slip factor


In [3]:
from pathlib import Path

# Robust path handling: try workspace-relative and absolute paths
possible_paths = [
    Path("Excerise_1") / "Datasets" / "Formation Tops.xlsx",
    Path("./Excerise_1") / "Datasets" / "Formation Tops.xlsx",
    Path("/workspaces/well_data_preprocessing/Excerise_1/Datasets/Formation Tops.xlsx")
]

tops_path = None
for p in possible_paths:
    if p.exists():
        tops_path = p
        break

if tops_path is None:
    raise FileNotFoundError(
        "Could not find 'Formation Tops.xlsx'. Expected one of: " +
        ", ".join(str(p) for p in possible_paths)
    )

df_tops = pd.read_excel(tops_path)
df_tops["Top depth [mMD_RKB]"] = df_tops["Top depth [mMD_RKB]"].astype("float64")

display(df_tops.head())
print("Loaded tops from:", tops_path)

,Well #,Formation,Top depth [mMD_RKB]
0,XX-X,FM-1,406.0
1,XX-X,FM-2,478.0
2,XX-X,FM-3,639.0
3,XX-X,FM-4,674.0
4,XX-X,FM-5,702.0


Loaded tops from: /workspaces/well_data_preprocessing/Excerise_1/Datasets/Formation Tops.xlsx


In [4]:
z_top10 = float(df_tops.loc[df_tops["Formation"]=="FM-10", "Top depth [mMD_RKB]"].iloc[0])
z_top11 = float(df_tops.loc[df_tops["Formation"]=="FM-11", "Top depth [mMD_RKB]"].iloc[0])
z0, z1 = z_top10, z_top11

print(f"FM-10 interval (mMD_RKB): {z0:.2f} to {z1:.2f}")

FM-10 interval (mMD_RKB): 1517.00 to 2141.00


In [5]:
possible_las_paths = [
    Path("Excerise_1") / "Datasets" / "Well Log.LAS",
    Path("./Excerise_1") / "Datasets" / "Well Log.LAS",
    Path("/workspaces/well_data_preprocessing/Excerise_1/Datasets/Well Log.LAS"),
    Path("/mnt/data/Well Log.LAS"),
]

las_path = None
for p in possible_las_paths:
    if p.exists():
        las_path = p
        break

if las_path is None:
    raise FileNotFoundError(
        "Could not find 'Well Log.LAS'. Expected one of:\n" +
        "\n".join(str(p) for p in possible_las_paths)
    )

las = lasio.read(las_path)
df_logs = las.df()
df_logs = df_logs.replace(-999.25, np.nan)
df_logs.index.name = "DEPT"
df_logs.head()

display(df_logs.head())
print("Loaded LAS from:", las_path)
print("Curves:", list(df_logs.columns))


,AC,ACS,BS,CALI,DEN,DENC,GR,K,NEU,PEF,RDEP,RMED,RMIC,ROP,TH,U
DEPT,,,,,,,,,,,,,,,,
405.9936,NaN,NaN,26.0,NaN,NaN,NaN,78.5885,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
406.1460,NaN,NaN,26.0,NaN,NaN,NaN,84.2673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
406.2984,NaN,NaN,26.0,NaN,NaN,NaN,83.9706,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
406.4508,NaN,NaN,26.0,NaN,NaN,NaN,83.9938,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
406.6032,NaN,NaN,26.0,NaN,NaN,NaN,76.8121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Loaded LAS from: /workspaces/well_data_preprocessing/Excerise_1/Datasets/Well Log.LAS
Curves: ['AC', 'ACS', 'BS', 'CALI', 'DEN', 'DENC', 'GR', 'K', 'NEU', 'PEF', 'RDEP', 'RMED', 'RMIC', 'ROP', 'TH', 'U']


In [6]:
z_top10 = float(df_tops.loc[df_tops["Formation"]=="FM-10", "Top depth [mMD_RKB]"].iloc[0])
z_top11 = float(df_tops.loc[df_tops["Formation"]=="FM-11", "Top depth [mMD_RKB]"].iloc[0])
z0, z1 = z_top10, z_top11

print(f"FM-10 interval (mMD_RKB): {z0:.2f} to {z1:.2f}")

FM-10 interval (mMD_RKB): 1517.00 to 2141.00


In [7]:
# --- Velocity from sonic (AC, ACS in us/ft) -> m/s ---
# V = (1e6 / slowness_us_per_ft) * 0.3048
df_logs["Vp"] = (1e6 / df_logs["AC"])  * 0.3048
df_logs["Vs"] = (1e6 / df_logs["ACS"]) * 0.3048

# --- Porosity ---
# Keep density in g/cc (DEN is usually g/cc in LAS)
rho_ma = 2.65 # g/cc (quartz-ish)
rho_f  = 1.00 # g/cc (brine-ish)

df_logs["phi_D"] = (rho_ma - df_logs["DEN"]) / (rho_ma - rho_f)  # v/v
df_logs["phi_N"] = df_logs["NEU"]                               # usually v/v
df_logs["phi"]   = df_logs[["phi_D","phi_N"]].mean(axis=1)

# Clip to physical range
df_logs.loc[(df_logs["phi"] < 0) | (df_logs["phi"] > 0.60), "phi"] = np.nan

# --- Bulk modulus K [GPa] using rho in g/cc and V in m/s ---
# K[GPa] = 1e-3 * rho(g/cc) * (Vp^2 - 4/3 Vs^2)
df_logs["Kbulk_GPa"] = 1e-6 * df_logs["DEN"] * (df_logs["Vp"]**2 - (4.0/3.0)*df_logs["Vs"]**2)

# density to SI
df_logs["rho_kgm3"] = df_logs["DEN"] * 1000.0

# --- Shale volume Vsh from GR (0–1) ---
GRmin = df_logs["GR"].quantile(0.05)
GRmax = df_logs["GR"].quantile(0.95)

# unngå deling på ~0 hvis GR er flat
if (GRmax - GRmin) == 0:
    df_logs["Vsh"] = np.nan
else:
    df_logs["Vsh"] = np.clip((df_logs["GR"] - GRmin) / (GRmax - GRmin), 0, 1)

# shear modulus (Pa)
df_logs["Mu_Pa"] = df_logs["rho_kgm3"] * (df_logs["Vs"]**2)

# convert
df_logs["Mu_GPa"] = df_logs["Mu_Pa"] / 1e9

# Basic sanity stats
display(df_logs[["Vp","Vs","DEN","phi","Kbulk_GPa", "Mu_GPa", "GR", "Vsh"]].describe())


,Vp,Vs,DEN,phi,Kbulk_GPa,Mu_GPa,GR,Vsh
count,9902.000000,9874.000000,9591.000000,9660.000000,9591.000000,9591.000000,11834.000000,11834.000000
mean,3376.580235,1756.009195,2.510800,0.146085,18.430447,7.961147,108.183922,0.478084
std,326.272733,276.538553,0.096185,0.039517,3.313977,2.522881,25.675389,0.270540
min,2334.303152,832.306831,2.101100,0.007171,5.660927,1.671153,41.161500,0.000000
25%,3184.851477,1579.379563,2.464300,0.122106,16.644970,6.196141,94.010750,0.298920
50%,3380.851605,1746.006969,2.541400,0.139523,18.531117,7.715325,107.998900,0.490508
75%,3572.791226,1934.759012,2.580600,0.158708,20.240718,9.475572,118.692525,0.636972
max,5714.232150,2932.788343,2.764100,0.392200,57.603265,22.807928,540.817400,1.000000


In [8]:
mask = df_logs[["Vp","phi","Vsh"]].notna().all(axis=1)
print("Min Vp (valid rows):", df_logs.loc[mask, "Vp"].min())
print("Any NaN left in inputs?", df_logs[["Vp","phi","Vsh"]].isna().sum())
print("Count Vp<=0:", (df_logs["Vp"] <= 0).sum())


Min Vp (valid rows): 2334.303151539009
Any NaN left in inputs? Vp     1938
phi    2180
Vsh       6
dtype: int64
Count Vp<=0: 0


In [9]:

# estimate cement:
import numpy as np

vcem_seeds = np.array([0, 0.005, 0.01, 0.02, 0.03, 0.04, 0.1])
phib_p = [0.3, 0.37, 0.38, 0.39, 0.395]  # define cement porosity for Vp

# --- Compute elastic bounds (model curves) ---
phi, vp1, vp2, vp3, vs1, vs2, vs3 = QI.screening(
    Dqz, Kqz, Gqz,
    Dsh, Ksh, Gsh,
    Dc,  Kc,  Gc,
    Db,  Kb,
    phib, phi_c,
    sigma, vsh,
    scheme, f, Cn
)

# --- CLEAN INPUTS BEFORE estimate_cem (prevents "zero-size array" crash) ---
df_qi = (
    df_logs[["Vp", "phi", "Vsh"]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

df_qi["Vp"] = df_qi["Vp"] / 1000.0   # m/s → km/s
df_qi = df_qi[df_qi["Vp"] > 1.0]

print(f"Rows used for QI: {len(df_qi)} / {len(df_logs)}")
print(f"Vp range used: {df_qi['Vp'].min():.2f} to {df_qi['Vp'].max():.2f}")

# --- Create QI object with CLEAN data ---
qi = QI(
    df_qi["Vp"].to_numpy(copy=True),
    phi=df_qi["phi"].to_numpy(copy=True),
    Vsh=df_qi["Vsh"].to_numpy(copy=True),
    sigma=sigma,
    phib=phib,
    f=f
)

# --- Estimate cement volume ---
vcem = qi.estimate_cem(
    vcem_seeds,
    Kqz, Gqz,
    Ksh, Gsh,
    phi_c, Cn,
    Kc, Gc,
    Db, Kb,
    scheme, vsh,
    Dsh, Dqz, Dc
)


Rows used for QI: 9637 / 11840
Vp range used: 2.33 to 5.71


ValueError: assignment destination is read-only

In [10]:
# Diagnostic: Check data availability before QI operations
print("=== Data Availability Check ===")
print(f"Total rows in df_logs: {len(df_logs)}")

# Check each required column
required_cols = ["Vp", "phi", "GR", "DEN", "Vs"]
for col in required_cols:
    valid_count = df_logs[col].notna().sum()
    print(f"{col}: {valid_count} valid values out of {len(df_logs)} ({100*valid_count/len(df_logs):.1f}%)")

# Check the mask that QI will use
valid_mask = df_logs[["Vp", "phi", "GR"]].notna().all(axis=1)
print(f"\nRows with all Vp, phi, GR valid: {valid_mask.sum()} out of {len(df_logs)}")

if valid_mask.sum() == 0:
    print("\n⚠️  WARNING: No valid rows! The arrays will be empty.")
    print("Check your input data for NaN values or calculation errors.")
else:
    print(f"\n✓ OK: {valid_mask.sum()} rows available for QI operations")


=== Data Availability Check ===
Total rows in df_logs: 11840
Vp: 9902 valid values out of 11840 (83.6%)
phi: 9660 valid values out of 11840 (81.6%)
GR: 11834 valid values out of 11840 (99.9%)
DEN: 9591 valid values out of 11840 (81.0%)
Vs: 9874 valid values out of 11840 (83.4%)

Rows with all Vp, phi, GR valid: 9637 out of 11840

✓ OK: 9637 rows available for QI operations
